In [ ]:
print("Hello")

In [1]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

# -------------------------------------------------------------
# Configuration
# -------------------------------------------------------------
SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
IMG_SIZE = 224
NUM_CLASSES = 6
MODEL_NAME = 'deit_small_patch16_224'

# Windows Best Practice: Wrap execution to prevent multiprocessing crashes
if __name__ == '__main__':
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    print(f"Active Device: {DEVICE}")

    # -------------------------------------------------------------
    # Hardcoded Local Dataset Path
    # -------------------------------------------------------------
    DATASET_DIR = r"C:\Users\LENOVO\Desktop\vks\ALL PROJECTS\crop-disease-detection\DATASETS\Rice_Leaf_AUG"
    print(f"Dataset root: {DATASET_DIR}")

    # -------------------------------------------------------------
    # Dataset Setup & Stratified Split
    # -------------------------------------------------------------
    class RiceLeafDataset(Dataset):
        def __init__(self, samples, transform=None):
            self.samples = samples
            self.transform = transform

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            path, label = self.samples[idx]
            image = Image.open(path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label

    # Identify classes
    class_names = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
    class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

    all_samples = []
    for cls_name in class_names:
        cls_folder = os.path.join(DATASET_DIR, cls_name)
        for img_file in os.listdir(cls_folder):
            if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_samples.append((os.path.join(cls_folder, img_file), class_to_idx[cls_name]))

    print(f"Total samples found: {len(all_samples)} across {len(class_names)} classes: {class_names}")

    labels = [s[1] for s in all_samples]
    train_samples, val_samples = train_test_split(
        all_samples, test_size=0.2, stratify=labels, random_state=SEED
    )

    # -------------------------------------------------------------
    # Data Transforms & Loaders
    # -------------------------------------------------------------
    train_transforms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = RiceLeafDataset(train_samples, transform=train_transforms)
    val_dataset = RiceLeafDataset(val_samples, transform=val_transforms)

    # Windows modification: num_workers set to 0
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    # -------------------------------------------------------------
    # Model Initialization
    # -------------------------------------------------------------
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    # -------------------------------------------------------------
    # Training & Validation Loop
    # -------------------------------------------------------------
    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print(f"\n--- Training {MODEL_NAME} ---")
    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss, running_corrects, total_train = 0.0, 0, 0
        
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == targets.data).item()
            total_train += targets.size(0)
            
        scheduler.step()
        
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = running_corrects / total_train
        
        # Validation
        model.eval()
        val_loss, val_corrects, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == targets.data).item()
                total_val += targets.size(0)
                
        epoch_val_loss = val_loss / total_val
        epoch_val_acc = val_corrects / total_val
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] "
              f"| Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.2f}% "
              f"| Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:.2f}%")
        
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), 'best_deit_rice_model.pth')
            print(f"  --> Saved new best model ({best_val_acc*100:.2f}%)")

    elapsed = time.time() - start_time
    print(f"\nCompleted in {elapsed//60:.0f}m {elapsed%60:.0f}s. Peak Val Accuracy: {best_val_acc*100:.2f}%")

    # -------------------------------------------------------------
    # Evaluation Report & Plotting
    # -------------------------------------------------------------
    model.load_state_dict(torch.load('best_deit_rice_model.pth'))
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())

    print("\n--- Final Classification Report ---")
    print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))

    # Plot training curves
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, NUM_EPOCHS + 1), history['train_loss'], label='Train')
    plt.plot(range(1, NUM_EPOCHS + 1), history['val_loss'], label='Val')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, NUM_EPOCHS + 1), history['train_acc'], label='Train')
    plt.plot(range(1, NUM_EPOCHS + 1), history['val_acc'], label='Val')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('deit_training_metrics.png')
    plt.show()

ModuleNotFoundError: No module named 'torch'